# `cuda_oracle.ipynb` — Kaggle CUDA authoritative oracle (BENCH-01 / BENCH-02)

**This is the authoritative GPU oracle of record.** It is a *human-gated external run*:
a human executes this notebook top-to-bottom on a **Kaggle CUDA** instance and records
the measured pass/fail + speed into `bench/RESULTS.md`. ROCm in-env is smoke-only — **not**
a gate.

## Harness order (correctness is a BLOCKING gate before any speed number)

1. Build the `--features cuda` wheel + `nvidia-smi` (confirm CUDA active).
2. **Standalone primitive oracles** (scan, segmented scan, radix sort / single-bit
   reorder, reduce-by-key, segmented-reduce, `update_part_props`) vs serial CPU refs —
   `<=1e-4`; integer/index primitives **bit-exact**.
3. **Bit-packed cindex oracle** vs CPU quantized layout (GPUT-15) — **bit-exact**.
4. **Depth-1 RMSE + Logloss** device-vs-CPU tree oracle — `<=1e-5` (Cosine score,
   first-order `calc_average` leaves).
5. **Steps 2–4 are ALL BLOCKING** — a failure halts the notebook so no fast-but-wrong
   speed number is ever reported (T-10-25).
6. Only then: **warm one untimed fit → time train-only**, draining CubeCL's lazy queue
   with a read-back/predict before stopping the clock → device vs host-CPU baseline
   (+ vs official CatBoost `task_type='GPU'` where a comparable depth-1 config exists).
7. Structured report → recorded by the human into `bench/RESULTS.md`.

## D-10-09 ESCALATION (read before trusting any depth-1 speed number)

> **Depth-1 device >= CPU is achievable ONLY at large n (~1e5–1e6+ rows); at small n it
> is physically infeasible regardless of optimization** — a depth-1 stump is the single
> most launch-overhead-bound workload in the milestone (a handful of kernel launches over
> trivial per-object work). The CPU grows it in microseconds; no fusion/residency makes a
> GPU launch + driver round-trip competitive at small n. **This is physics, not a tuning
> gap.** BENCH-02's depth-1 speed bar is therefore PINNED to the large-n synthetic
> workload (D-06 `SPEED_CONFIG`, ~1e6×50, tunable above break-even). **The Kaggle run is
> the arbiter of the crossover.** If device still loses even at large n after fused/
> resident optimization, record the measured crossover (or its absence) and let the user
> decide whether D-10-09 stands for depth-1 (depth-6 in Phase 11 is where device
> dominance is unambiguous). **Do NOT fabricate numbers** — every measured cell below is
> filled by the actual run.

## Step 0 — build the `--features cuda` wheel + confirm CUDA

Repo is expected checked out at `REPO`. Adjust the path for the Kaggle working dir.

In [ ]:
import os, subprocess, sys, time, json
import numpy as np

# --- repo layout (adjust REPO for the Kaggle working dir) -----------------------------
REPO = os.environ.get('CATBOOST_RS_REPO', os.getcwd())
BENCH = os.path.join(REPO, 'bench')
FIXTURES = os.path.join(BENCH, 'fixtures')
sys.path.insert(0, BENCH)
import generator  # D-06 single source: correctness fixture AND large-n speed workload

def run(cmd, cwd=REPO, check=True):
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    sys.stdout.write(r.stdout);  sys.stderr.write(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {" ".join(cmd)}')
    return r

# Build the CUDA wheel (device path) and install it.
run(['maturin', 'build', '--release', '--no-default-features', '--features', 'cuda',
     '-m', 'crates/cb-python/Cargo.toml'])
# pip-install the freshly built wheel (newest by mtime under target/wheels).
wheels = sorted(
    (os.path.join(dp, f) for dp, _, fs in os.walk(os.path.join(REPO, 'target', 'wheels'))
     for f in fs if f.endswith('.whl')),
    key=os.path.getmtime)
assert wheels, 'no wheel built under target/wheels'
run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', wheels[-1]])

In [ ]:
# nvidia-smi — CUDA MUST be active or the whole run is meaningless.
smi = run(['nvidia-smi'], check=False)
assert smi.returncode == 0, 'nvidia-smi failed: this notebook must run on a CUDA instance'
print('CUDA confirmed active.')

## Steps 2–4 — BLOCKING correctness gate

The standalone primitive + cindex oracles run as the in-tree Rust device tests under
`--features cuda` (the same oracles that run on ROCm in-env, now on CUDA). The depth-1
RMSE/Logloss oracle compares the device `fit()` against the committed CPU-path expected
values in `bench/fixtures/`. **Any failure raises and halts — the speed cells assert the
gate passed before timing anything (T-10-25).**

In [ ]:
# GATE flag: set True only if ALL correctness oracles pass. Speed cells assert this.
GATE_PASSED = False
gate_report = {}

# --- Step 2 + 3: standalone primitive + cindex oracles (Rust device tests) ------------
# These encode the <=1e-4 / bit-exact bars in-tree (SPIKE-REDUCTION.md reduce harness et al).
prim = run(['cargo', 'test', '--no-default-features', '--features', 'cuda',
            '-p', 'cb-backend', '--', '--nocapture',
            'scan', 'segmented', 'sort', 'reorder', 'reduce', 'cindex', 'update_part_props'],
           check=False)
# --nocapture surfaces the reduce-strategy variance/path + err lines
# (reduce_finalize_strategies_are_deterministic_and_report_path). Transcribe the CUDA
# err/ms for candidates (a) FixedOrderTree / (b) HostSum / (c) FixedPointAtomic into
# SPIKE-REDUCTION.md section 4 from this output.
gate_report['primitive_cindex_oracles'] = 'PASS' if prim.returncode == 0 else 'FAIL'
assert prim.returncode == 0, 'BLOCKING: primitive/cindex device oracles FAILED — halting.'

In [ ]:
# --- Step 4: depth-1 RMSE + Logloss device-vs-CPU-reference oracle (<=1e-5) ------------
# Load committed correctness fixtures + numpy CPU-path expected values (D-06 single source).
X = np.load(os.path.join(FIXTURES, 'X_small.npy'))
y_reg = np.load(os.path.join(FIXTURES, 'y_small_reg.npy'))
y_bin = np.load(os.path.join(FIXTURES, 'y_small_bin.npy'))
with open(os.path.join(FIXTURES, 'expected_depth1_tree.json')) as fh:
    expected = json.load(fh)
lr = expected['learning_rate']

from catboost_rs import CatBoostRegressor, CatBoostClassifier  # device wheel (cuda)

EPS = 1e-5

# RMSE depth-1: one tree, first-order calc_average leaves. Device predictions on the
# training set for a single depth-1 tree == the per-object leaf value; compare against
# the committed serial reference (leaf_left/leaf_right on the reference split).
reg = CatBoostRegressor(iterations=1, depth=1, learning_rate=lr, l2_leaf_reg=expected['l2'])
reg.fit(X, y_reg)
pred_reg = np.asarray(reg.predict(X), dtype=np.float64)
ref = expected['rmse']
col = X[:, ref['best_feature']].astype(np.float64)
ref_pred = np.where(col > ref['best_border'], ref['leaf_right'], ref['leaf_left'])
rmse_err = float(np.max(np.abs(pred_reg - ref_pred)))
print(f'depth-1 RMSE device-vs-reference max abs err = {rmse_err:.3e} (bar {EPS:.0e})')
gate_report['depth1_rmse_max_abs_err'] = rmse_err
assert rmse_err <= EPS, f'BLOCKING: depth-1 RMSE oracle {rmse_err:.3e} > {EPS:.0e} — halting.'

# Logloss depth-1: FIRST-ORDER calc_average leaves (NOT Newton der2 — Phase 11 / GPUT-07).
clf = CatBoostClassifier(iterations=1, depth=1, learning_rate=lr, l2_leaf_reg=expected['l2'],
                         loss_function='Logloss')
clf.fit(X, y_bin)
raw = np.asarray(clf.predict(X, prediction_type='RawFormulaVal'), dtype=np.float64)
refl = expected['logloss']
coll = X[:, refl['best_feature']].astype(np.float64)
ref_raw = np.where(coll > refl['best_border'], refl['leaf_right'], refl['leaf_left'])
ll_err = float(np.max(np.abs(raw - ref_raw)))
print(f'depth-1 Logloss device-vs-reference max abs err = {ll_err:.3e} (bar {EPS:.0e})')
gate_report['depth1_logloss_max_abs_err'] = ll_err
assert ll_err <= EPS, f'BLOCKING: depth-1 Logloss oracle {ll_err:.3e} > {EPS:.0e} — halting.'

GATE_PASSED = True
print('\n=== CORRECTNESS GATE PASSED — speed cells unlocked ===')

## Steps 5–6 — warm-run / JIT-excluded train-only speed (large-n workload ONLY)

Pinned to the D-06 large-n `SPEED_CONFIG` (~1e6×50) per the D-10-09 escalation above.
**Warm one untimed fit** (absorbs JIT), then **time train-only** and force a read-back/
predict to **drain CubeCL's lazy queue** before stopping the clock (else the timer
under-reports; CubeCL executes lazily — manual `05_lazy_execution.md`,
`11_launch_overhead_and_transfers.md`).

In [ ]:
assert GATE_PASSED, 'REFUSING to time speed: correctness gate did not pass (T-10-25).'

# Large-n workload from the SAME seeded generator that produced the correctness fixture.
Xl, yl = generator.generate(**generator.SPEED_CONFIG)
print(f'speed workload: {Xl.shape[0]} rows x {Xl.shape[1]} features (D-06 SPEED_CONFIG)')

ITERS = 100  # depth-1 boosting rounds; tune so the device histogram amortizes launch cost

def timed_fit(make_model):
    m = make_model()
    m.fit(Xl, yl)               # WARM (untimed) — absorbs JIT / first-launch compile
    m2 = make_model()
    t0 = time.time()
    m2.fit(Xl, yl)
    _ = np.asarray(m2.predict(Xl[:1024]))  # DRAIN the lazy queue before stopping the clock
    return time.time() - t0

device_s = timed_fit(lambda: CatBoostRegressor(iterations=ITERS, depth=1, learning_rate=0.1))
print(f'device (cuda) train-only wall-clock: {device_s:.4f}s')
gate_report['speed_device_s'] = device_s

In [ ]:
# Host-CPU baseline + official CatBoost GPU, where available on the box.
# NOTE: the catboost_rs CPU baseline requires a SEPARATE cpu-feature wheel (features are
# compile-time, CLAUDE.md). Build+install it in a fresh cell / kernel if comparing on the
# same box; otherwise record the CPU number from a cpu-wheel run and paste into RESULTS.md.
cpu_s = None      # <-- fill from a cpu-feature wheel run (see note above)
catboost_gpu_s = None
try:
    import catboost as _cb
    m = _cb.CatBoostRegressor(iterations=ITERS, depth=1, learning_rate=0.1,
                              task_type='GPU', verbose=False)
    m.fit(Xl, yl); m2 = _cb.CatBoostRegressor(iterations=ITERS, depth=1, learning_rate=0.1,
                                              task_type='GPU', verbose=False)
    t0 = time.time(); m2.fit(Xl, yl); _ = m2.predict(Xl[:1024])
    catboost_gpu_s = time.time() - t0
    print(f'official CatBoost GPU train-only: {catboost_gpu_s:.4f}s')
except Exception as e:
    print('official CatBoost GPU comparison skipped:', e)
gate_report['speed_cpu_s'] = cpu_s
gate_report['speed_catboost_gpu_s'] = catboost_gpu_s

### Fill `SPIKE-REDUCTION.md` §4 (CUDA-only reduce rows) from this run

The `--nocapture` reduce output above reports, per finalize strategy, the run-to-run
spread + path (and `abs_div` vs the f64 baseline). Transcribe the CUDA `err (<=1e-4)` and
`ms` for the three reduce candidates into `SPIKE-REDUCTION.md` §4:

| Candidate | CUDA err (<=1e-4) | CUDA ms |
|-----------|-------------------|---------|
| (a) Fixed-order tree reduce | _paste_ | _paste_ |
| (b) Block-then-host-sum     | _paste_ | _paste_ |
| (c) Fixed-point u64 atomics | _paste_ | _paste_ |

These are **not fabricated** — they come straight from the notebook's device run.

## Step 7 — structured report

Copy the printed rows into `bench/RESULTS.md` (correctness pass/fail + speed) and fill the
`SPIKE-REDUCTION.md` §4 CUDA `err`/`ms` rows from the reduce-strategy oracle output above.
Correctness rows come FIRST; the speed row is only meaningful because the gate passed.

In [ ]:
print('================ CUDA ORACLE STRUCTURED REPORT ================')
print('CORRECTNESS (blocking gate):')
print(f"  primitive + cindex oracles : {gate_report.get('primitive_cindex_oracles')}")
print(f"  depth-1 RMSE   max|err|     : {gate_report.get('depth1_rmse_max_abs_err'):.3e}  (bar 1e-5)")
print(f"  depth-1 Logloss max|err|    : {gate_report.get('depth1_logloss_max_abs_err'):.3e}  (bar 1e-5)")
print(f"  GATE PASSED                 : {GATE_PASSED}")
print('SPEED (large-n SPEED_CONFIG, warm-run/JIT-excluded, queue drained):')
print(f"  device (cuda)   train-only  : {gate_report.get('speed_device_s')} s")
print(f"  host-CPU baseline           : {gate_report.get('speed_cpu_s')} s  (cpu-wheel; see note)")
print(f"  official CatBoost GPU        : {gate_report.get('speed_catboost_gpu_s')} s")
print('D-10-09: depth-1 device>=CPU only at large n; small n physically infeasible.')
print('==============================================================')
print(json.dumps(gate_report, indent=2, default=str))